# 🛍️ Project 10 — Customer Segmentation with K-Means Clustering

---

## Learning Goals
By the end of this notebook you will be able to:
- Explain what **unsupervised learning** is and when to use it
- Describe how the **K-Means algorithm** works step by step
- Use the **Elbow Method** to pick the right number of clusters
- **Scale features** before applying distance-based algorithms
- Interpret a **silhouette score** to evaluate cluster quality
- Visualise multi-dimensional clusters in 2-D scatter plots

## Estimated Time
⏱️ **45 – 60 minutes** (reading + running + experimenting)

## Prerequisites
- Basic Python (lists, loops, functions)
- Familiarity with `pandas` DataFrames and `matplotlib` plots

---

## Background — What is Customer Segmentation?

Retailers collect data about customers: how much they earn, how often they shop, and how much they spend. **Segmentation** means grouping customers who behave similarly, so the business can tailor offers to each group.

This is an **unsupervised learning** problem — nobody has labelled the customers in advance. The algorithm must discover the groups on its own.

### How K-Means Works (in plain English)
1. **Initialise** — randomly place *k* centroids in feature space.
2. **Assign** — assign each data point to the nearest centroid.
3. **Update** — move each centroid to the mean position of all its assigned points.
4. **Repeat** steps 2–3 until centroids stop moving (convergence).

> 💡 **Intuition**: Imagine dropping five pins on a map and then grouping every house to its nearest pin. After each round you move each pin to the centre of its group. Repeat until the pins stop moving.

## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Make plots look nicer
sns.set_theme(style="whitegrid")
print("Libraries imported successfully ✅")

## Step 2 — Generate a Synthetic Customer Dataset

We create 300 fictional customers with two attributes:
- **AnnualIncome_k** — annual income in thousands of dollars
- **SpendingScore** — a score from 1 (never buys) to 100 (buys everything)

We embed five natural clusters so we can later verify our algorithm found them.

In [ ]:
def generate_customer_data(n_samples=300, random_state=42):
    """Return a synthetic customer DataFrame with five natural clusters."""
    rng = np.random.default_rng(random_state)

    # Five cluster centres: [income, spending_score]
    centres = [
        [25, 78],   # Low income, high spenders
        [55, 55],   # Mid income, mid spenders
        [85, 20],   # High income, low spenders
        [85, 82],   # High income, high spenders
        [30, 25],   # Low income, low spenders
    ]
    cluster_sizes = [n_samples // 5] * 4 + [n_samples - 4 * (n_samples // 5)]

    records = []
    for centre, size in zip(centres, cluster_sizes):
        income = rng.normal(loc=centre[0], scale=8, size=size).clip(15, 135)
        score  = rng.normal(loc=centre[1], scale=8, size=size).clip(1, 100)
        for i_val, s_val in zip(income, score):
            records.append({"AnnualIncome_k": round(i_val, 1),
                             "SpendingScore": round(s_val, 1)})

    df = pd.DataFrame(records)
    df.insert(0, "CustomerID", range(1, len(df) + 1))
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)

df = generate_customer_data(n_samples=300)
print(f"Dataset shape: {df.shape}")
df.head(10)

### Quick Exploration

In [ ]:
print(df.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["AnnualIncome_k"].hist(bins=20, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of Annual Income")
axes[0].set_xlabel("Annual Income (k$)")

df["SpendingScore"].hist(bins=20, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Distribution of Spending Score")
axes[1].set_xlabel("Spending Score")

plt.tight_layout()
plt.show()

## Step 3 — Feature Scaling

K-Means measures **Euclidean distance** between points. If income ranges from 15 to 135 and spending ranges from 1 to 100, the income axis has more "spread" and will dominate the distance calculation.

**StandardScaler** transforms each feature to have **mean = 0** and **standard deviation = 1**, putting both features on equal footing.

```
scaled_value = (original_value - mean) / std_deviation
```

In [ ]:
feature_cols = ["AnnualIncome_k", "SpendingScore"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# Verify: means should be ~0, stds should be ~1
print("After scaling:")
print(f"  Mean  → {X_scaled.mean(axis=0).round(4)}")
print(f"  StdDev → {X_scaled.std(axis=0).round(4)}")

## Step 4 — Choosing k with the Elbow Method

K-Means requires you to specify **k** (the number of clusters) upfront. But how do you pick the right k?

The **Elbow Method**:
- Run K-Means for k = 2, 3, 4, … 10
- Record the **inertia** (sum of squared distances to nearest centroid)
- Plot inertia vs k — look for the "elbow" where improvement slows down

We also track the **Silhouette Score** — values closer to +1 mean clusters are well-separated.

In [ ]:
k_range = range(2, 11)
inertias    = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Show results table
results_df = pd.DataFrame({
    "k": list(k_range),
    "Inertia": [round(x, 1) for x in inertias],
    "Silhouette": [round(x, 4) for x in silhouettes]
})
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow curve
axes[0].plot(list(k_range), inertias, marker="o", color="steelblue", linewidth=2)
axes[0].axvline(x=5, color="tomato", linestyle="--", label="Chosen k=5")
axes[0].set_title("Elbow Method — Inertia vs k")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Silhouette scores
axes[1].plot(list(k_range), silhouettes, marker="s", color="seagreen", linewidth=2)
axes[1].axvline(x=5, color="tomato", linestyle="--", label="Chosen k=5")
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Choosing the Right Number of Clusters", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print("➡️  k=5 shows a clear elbow and a high silhouette score.")

## Step 5 — Fit K-Means with k = 5

In [ ]:
# Fit final model
km_final = KMeans(n_clusters=5, random_state=42, n_init=10)
km_final.fit(X_scaled)

# Assign cluster labels back to the original DataFrame
df["Cluster"] = km_final.labels_

final_sil = silhouette_score(X_scaled, km_final.labels_)
print(f"Final Silhouette Score (k=5): {final_sil:.4f}")
print(f"Cluster sizes:\n{df['Cluster'].value_counts().sort_index()}")

## Step 6 — Visualise the Clusters

In [ ]:
palette = sns.color_palette("tab10", n_colors=5)

plt.figure(figsize=(9, 6))
for cluster_id, group in df.groupby("Cluster"):
    plt.scatter(
        group["AnnualIncome_k"], group["SpendingScore"],
        label=f"Cluster {cluster_id}",
        color=palette[cluster_id], alpha=0.8, s=70,
        edgecolors="white", linewidths=0.5
    )

plt.title("Customer Segments — Annual Income vs Spending Score", fontsize=14)
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1–100)", fontsize=12)
plt.legend(title="Cluster", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("customer_segments.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to customer_segments.png")

## Step 7 — Interpret the Clusters

Let's compute the average income and spending score for each cluster to understand what each group represents.

In [ ]:
summary = (
    df.groupby("Cluster")
      .agg(
          Count=("CustomerID", "count"),
          Avg_Income=("AnnualIncome_k", "mean"),
          Avg_Spending=("SpendingScore", "mean")
      )
      .round(1)
      .reset_index()
)

# Sort for easier reading
summary = summary.sort_values(["Avg_Income", "Avg_Spending"]).reset_index(drop=True)
print(summary.to_string(index=False))
print("\nInterpretation:")
print("  Low income + high spending  → Impulsive / lifestyle-driven shoppers")
print("  Low income + low spending   → Budget-conscious / needs-based shoppers")
print("  Mid income + mid spending   → Balanced / mainstream customers")
print("  High income + low spending  → Careful savers / non-spenders")
print("  High income + high spending → Premium / target customers for loyalty programs")

---

## 🧪 Try It Yourself

Here are some experiments to deepen your understanding:

1. **Change k** — Refit with k=3 or k=7. How do the clusters look? Does the silhouette score go up or down?

2. **Remove scaling** — Run K-Means on the raw (unscaled) features. Does it still find the correct clusters?

3. **Add a third feature** — Uncomment the cell below to add an `Age` column and cluster in 3-D.

4. **Random initialisation** — Change `random_state=42` to `random_state=0` or `random_state=99`. Does the clustering change?

5. **Larger dataset** — Call `generate_customer_data(n_samples=1000)`. Does K-Means still converge quickly?

In [ ]:
# Experiment 1: Try k=3
km_k3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_k3 = km_k3.fit_predict(X_scaled)
sil_k3 = silhouette_score(X_scaled, labels_k3)
print(f"Silhouette Score with k=3: {sil_k3:.4f}")

# Try k=7
km_k7 = KMeans(n_clusters=7, random_state=42, n_init=10)
labels_k7 = km_k7.fit_predict(X_scaled)
sil_k7 = silhouette_score(X_scaled, labels_k7)
print(f"Silhouette Score with k=7: {sil_k7:.4f}")

print(f"\nBest k so far (k=5):       {final_sil:.4f}")
print("→ k=5 likely gives the best separation for this dataset.")

In [ ]:
# Experiment 2: What happens without scaling?
X_raw = df[["AnnualIncome_k", "SpendingScore"]].values
km_raw = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_raw = km_raw.fit_predict(X_raw)
sil_raw = silhouette_score(X_raw, labels_raw)
print(f"Silhouette Score WITHOUT scaling: {sil_raw:.4f}")
print(f"Silhouette Score WITH scaling:    {final_sil:.4f}")
print("\nNote: For this balanced dataset the difference may be small,")
print("but in real datasets with very different feature ranges, scaling matters a lot.")

---

## 📋 Summary — Concepts Learned

| Concept | Plain English Explanation |
|---|---|
| **Unsupervised Learning** | Finding patterns in data with no predefined labels |
| **K-Means Algorithm** | Groups data by repeatedly assigning points to nearest centroid and recomputing centroids |
| **Inertia** | Sum of squared distances from points to their cluster centre; lower = tighter clusters |
| **Elbow Method** | Plot inertia vs k; the "elbow" point suggests the optimal k |
| **Silhouette Score** | Measures how well-separated clusters are; ranges from -1 to +1 |
| **Feature Scaling** | Normalising feature ranges so distance calculations are not biased |
| **StandardScaler** | Transforms features to mean=0, std=1 using (x - mean) / std |
| **Cluster Centroid** | The mean position of all points in a cluster |

---

## Next Steps
- Try **DBSCAN** — a clustering algorithm that doesn't require specifying k upfront and can find non-spherical clusters.
- Explore **Hierarchical Clustering** for a tree-based view of customer segments.
- Apply this to a real dataset — the [Mall Customers Dataset](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) on Kaggle is a great starting point.

**Well done on completing Project 10! 🎉**